In [12]:
import os
from langgraph import graph
from langgraph.graph import StateGraph,END,START
from typing import Dict,TypedDict,List,Union,Annotated,Sequence
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage,ToolMessage,SystemMessage
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langchain_deepseek import ChatDeepSeek
from langgraph.prebuilt import ToolNode, tool_node

In [14]:
load_dotenv()

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]

@tool
def add(a:int, b:int) -> int:
    '将整数相加返回结果'
    return a+b

tools = [add]

llm = ChatDeepSeek(model="deepseek-chat", temperature=0).bind_tools(tools)

def model_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content= '你是一个ai助理，请认真回答我的问题。')
    response = llm.invoke ([system_prompt] + state['messages'])
    return {'messages':[response]}

def should_continue(state:AgentState) :
    messages = state['messages']
    last_message = messages[-1]
    if not last_message.tool_calls:
        return 'end'
    else:
        return 'continue'


In [19]:
graph = StateGraph(AgentState)
graph.add_node('agent_node', model_call)

tool_node = ToolNode(tools = tools)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'agent_node')
graph.add_conditional_edges('agent_node',should_continue,{'continue':'tools','end':END})
graph.add_edge('tools', 'agent_node')

agent = graph.compile()

In [21]:
def print_stream(stream):
    for s in stream:
        messages = s['messages'][-1]
        if isinstance(messages, tuple):
            print(messages)
        else:
            messages.pretty_print()

input = {'messages':[('user','add 15+20')]}
print_stream(agent.stream(input,stream_mode = 'values'))

================================ Human Message =================================

add 15+20
================================== Ai Message ==================================
Tool Calls:
  add (call_00_TLWPJk7KjGcaBGgJDky01115)
 Call ID: call_00_TLWPJk7KjGcaBGgJDky01115
  Args:
    a: 15
    b: 20
================================= Tool Message =================================
Name: add

35
================================== Ai Message ==================================

15 + 20 = **35**
